In [1]:
# 1. Importar bibliotecas

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# 2. Criar a SparkSession

spark = SparkSession.builder \
    .appName("Projeto Haiti") \
    .getOrCreate()

In [2]:
# 3. Ler o arquivo

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("atividades_em_missao_de_paz_no_haiti.CSV")

In [3]:
# 4. Visualizar os dados

df.show(5)

df.tail(5)

+---------------------------------------------+----+-------+-------+---------+---+---+-------+-------+----------+-------+
|CONTROLE DE EFETIVO NO HAITI AT� MAIO DE 2016| ANO|BRABAT1|BRABAT2|BRAENGCOY| MB|FAB|ESTRANG|SOMA EB|SOMA TOTAL|POR ANO|
+---------------------------------------------+----+-------+-------+---------+---+---+-------+-------+----------+-------+
|                                  1� CONTBRAS|2004|    949|      0|        0|250|  1|      0|    949|      1200|   NULL|
|                                  2� CONTBRAS|2004|    949|      0|        0|250|  1|      0|    949|      1200|   2400|
|                                  3� CONTBRAS|2005|    824|      0|      150|225|  1|      0|    974|      1200|   NULL|
|                                  4� CONTBRAS|2005|    824|      0|      150|225|  1|      0|    974|      1200|   2400|
|                                  5� CONTBRAS|2006|    822|      0|      150|225|  1|      0|    972|      1198|   NULL|
+-----------------------

[Row(CONTROLE DE EFETIVO NO HAITI AT� MAIO DE 2016='23� CONTBRAS', ANO=2015, BRABAT1=665, BRABAT2=0, BRAENGCOY=120, MB=181, FAB=4, ESTRANG=0, SOMA EB=785, SOMA TOTAL=970, POR ANO=1940),
 Row(CONTROLE DE EFETIVO NO HAITI AT� MAIO DE 2016='23� CONTBRAS', ANO=2016, BRABAT1=665, BRABAT2=0, BRAENGCOY=120, MB=181, FAB=4, ESTRANG=0, SOMA EB=785, SOMA TOTAL=970, POR ANO=None),
 Row(CONTROLE DE EFETIVO NO HAITI AT� MAIO DE 2016='24� CONTBRAS', ANO=2016, BRABAT1=665, BRABAT2=0, BRAENGCOY=120, MB=181, FAB=4, ESTRANG=0, SOMA EB=785, SOMA TOTAL=970, POR ANO=None),
 Row(CONTROLE DE EFETIVO NO HAITI AT� MAIO DE 2016='25� CONTBRAS', ANO=2016, BRABAT1=636, BRABAT2=0, BRAENGCOY=120, MB=181, FAB=30, ESTRANG=0, SOMA EB=756, SOMA TOTAL=967, POR ANO=2907),
 Row(CONTROLE DE EFETIVO NO HAITI AT� MAIO DE 2016='26� CONTBRAS', ANO=2017, BRABAT1=639, BRABAT2=0, BRAENGCOY=120, MB=181, FAB=30, ESTRANG=0, SOMA EB=759, SOMA TOTAL=970, POR ANO=970)]

In [4]:
# 5. Quantidade de linhas e colunas

print(f"Quantidade de linhas: {df.count()}")
print(f"Quantidade de colunas: {len(df.columns)}")

Quantidade de linhas: 27
Quantidade de colunas: 11


In [5]:
# 6. Nome das colunas

print(df.columns)

['CONTROLE DE EFETIVO NO HAITI AT� MAIO DE 2016', 'ANO', 'BRABAT1', 'BRABAT2', 'BRAENGCOY', 'MB', 'FAB', 'ESTRANG', 'SOMA EB', 'SOMA TOTAL', 'POR ANO']


In [6]:
# 7. Tipos das colunas

df.printSchema()

root
 |-- CONTROLE DE EFETIVO NO HAITI AT� MAIO DE 2016: string (nullable = true)
 |-- ANO: integer (nullable = true)
 |-- BRABAT1: integer (nullable = true)
 |-- BRABAT2: integer (nullable = true)
 |-- BRAENGCOY: integer (nullable = true)
 |-- MB: integer (nullable = true)
 |-- FAB: integer (nullable = true)
 |-- ESTRANG: integer (nullable = true)
 |-- SOMA EB: integer (nullable = true)
 |-- SOMA TOTAL: integer (nullable = true)
 |-- POR ANO: integer (nullable = true)



In [7]:
# 8. Verificação de dados faltantes

from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

# Os valores nulos presentes na coluna "POR ANO" representam linhas intermediárias dos
# contingentes, sendo preenchidos apenas ao final de cada ano. Portanto, não caracterizam erro de coleta.

+---------------------------------------------+---+-------+-------+---------+---+---+-------+-------+----------+-------+
|CONTROLE DE EFETIVO NO HAITI AT� MAIO DE 2016|ANO|BRABAT1|BRABAT2|BRAENGCOY| MB|FAB|ESTRANG|SOMA EB|SOMA TOTAL|POR ANO|
+---------------------------------------------+---+-------+-------+---------+---+---+-------+-------+----------+-------+
|                                            0|  0|      0|      0|        0|  0|  0|      0|      0|         0|     13|
+---------------------------------------------+---+-------+-------+---------+---+---+-------+-------+----------+-------+



In [8]:
# 9. Verificar registros duplicados

print("Total:", df.count())

print("Distintos:", df.distinct().count())

# Não foram encontrados registros duplicados.

Total: 27
Distintos: 27


In [9]:
# 10. Estatísticas descritivas

df.describe().show()

+-------+---------------------------------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|summary|CONTROLE DE EFETIVO NO HAITI AT� MAIO DE 2016|               ANO|           BRABAT1|           BRABAT2|         BRAENGCOY|                MB|               FAB|           ESTRANG|           SOMA EB|        SOMA TOTAL|           POR ANO|
+-------+---------------------------------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|  count|                                           27|                27|                27|                27|                27|                27|                27|                27|                27|                27|                14|
|   mean|       

In [10]:
# 11. Efetivo total enviado

from pyspark.sql.functions import sum

df.select(
    sum("SOMA TOTAL").alias("Total de militares")
).show()

+------------------+
|Total de militares|
+------------------+
|             37449|
+------------------+



In [11]:
# 12. Média de militares por contingente

from pyspark.sql.functions import avg

df.select(
    avg("SOMA TOTAL").alias("Média")
).show()

+------+
| Média|
+------+
|1387.0|
+------+



In [12]:
# 13. Maior contingente

df.orderBy(col("SOMA TOTAL").desc()).show(5)

+---------------------------------------------+----+-------+-------+---------+---+---+-------+-------+----------+-------+
|CONTROLE DE EFETIVO NO HAITI AT� MAIO DE 2016| ANO|BRABAT1|BRABAT2|BRAENGCOY| MB|FAB|ESTRANG|SOMA EB|SOMA TOTAL|POR ANO|
+---------------------------------------------+----+-------+-------+---------+---+---+-------+-------+----------+-------+
|                                 15� CONTBRAS|2011|    775|    809|      250|302| 30|     32|   1834|      2198|   4388|
|                                 12� CONTBRAS|2010|    796|    809|      250|309|  2|     32|   1855|      2198|   NULL|
|                                 13� CONTBRAS|2010|    794|    809|      250|309|  2|     32|   1853|      2196|   4394|
|                                 14� CONTBRAS|2011|    767|    809|      250|302| 30|     32|   1826|      2190|   NULL|
|                                 16� CONTBRAS|2012|    581|    768|      250|249| 30|     32|   1599|      1910|   NULL|
+-----------------------

In [13]:
# 14. Menor contingente

df.orderBy(col("SOMA TOTAL")).show(5)

+---------------------------------------------+----+-------+-------+---------+---+---+-------+-------+----------+-------+
|CONTROLE DE EFETIVO NO HAITI AT� MAIO DE 2016| ANO|BRABAT1|BRABAT2|BRAENGCOY| MB|FAB|ESTRANG|SOMA EB|SOMA TOTAL|POR ANO|
+---------------------------------------------+----+-------+-------+---------+---+---+-------+-------+----------+-------+
|                                 25� CONTBRAS|2016|    636|      0|      120|181| 30|      0|    756|       967|   2907|
|                                 23� CONTBRAS|2016|    665|      0|      120|181|  4|      0|    785|       970|   NULL|
|                                 22� CONTBRAS|2015|    665|      0|      120|181|  4|      0|    785|       970|   NULL|
|                                 23� CONTBRAS|2015|    665|      0|      120|181|  4|      0|    785|       970|   1940|
|                                 24� CONTBRAS|2016|    665|      0|      120|181|  4|      0|    785|       970|   NULL|
+-----------------------

In [14]:
# 15. Evolução anual

from pyspark.sql.functions import sum

df.groupBy("ANO") \
    .agg(
        sum("SOMA TOTAL").alias("TOTAL")
    ) \
    .orderBy("ANO") \
    .show()

+----+-----+
| ANO|TOTAL|
+----+-----+
|2004| 2400|
|2005| 2400|
|2006| 2395|
|2007| 2394|
|2008| 2494|
|2009| 1293|
|2010| 4394|
|2011| 4388|
|2012| 3820|
|2013| 2900|
|2014| 2754|
|2015| 1940|
|2016| 2907|
|2017|  970|
+----+-----+



In [15]:
# 16. Participação das forças

df.select(
    sum("BRABAT1").alias("BRABAT1"),
    sum("BRABAT2").alias("BRABAT2"),
    sum("BRAENGCOY").alias("BRAENGCOY"),
    sum("MB").alias("Marinha"),
    sum("FAB").alias("FAB"),
    sum("ESTRANG").alias("Estrangeiros")
).show()

+-------+-------+---------+-------+---+------------+
|BRABAT1|BRABAT2|BRAENGCOY|Marinha|FAB|Estrangeiros|
+-------+-------+---------+-------+---+------------+
|  20987|   4772|     4624|   6295|347|         424|
+-------+-------+---------+-------+---+------------+



In [16]:
# 17. Participação percentual

from pyspark.sql.functions import sum

totais = df.select(
    sum("BRABAT1").alias("BRABAT1"),
    sum("BRABAT2").alias("BRABAT2"),
    sum("BRAENGCOY").alias("BRAENGCOY"),
    sum("MB").alias("MB"),
    sum("FAB").alias("FAB")
)

totais.show()

+-------+-------+---------+----+---+
|BRABAT1|BRABAT2|BRAENGCOY|  MB|FAB|
+-------+-------+---------+----+---+
|  20987|   4772|     4624|6295|347|
+-------+-------+---------+----+---+



In [17]:
# 18. Total por ano

df.select("ANO", "POR ANO") \
    .filter(col("POR ANO").isNotNull()) \
    .show()

+----+-------+
| ANO|POR ANO|
+----+-------+
|2004|   2400|
|2005|   2400|
|2006|   2395|
|2007|   2394|
|2008|   2494|
|2009|   1293|
|2010|   4394|
|2011|   4388|
|2012|   3820|
|2013|   2900|
|2014|   2754|
|2015|   1940|
|2016|   2907|
|2017|    970|
+----+-------+



In [18]:
# 19. Quantidade de contingentes por ano

df.groupBy("ANO") \
.count() \
.orderBy("ANO") \
.show()

+----+-----+
| ANO|count|
+----+-----+
|2004|    2|
|2005|    2|
|2006|    2|
|2007|    2|
|2008|    2|
|2009|    1|
|2010|    2|
|2011|    2|
|2012|    2|
|2013|    2|
|2014|    2|
|2015|    2|
|2016|    3|
|2017|    1|
+----+-----+



In [19]:
# 20. Correlação

df.stat.corr("BRABAT1", "SOMA TOTAL")

0.0008507097904763671

In [20]:
df.stat.corr("BRAENGCOY", "SOMA TOTAL")

0.6933012755543149

In [21]:
df.stat.corr("MB", "SOMA TOTAL")

0.9159576635787996

In [22]:
df.stat.corr("FAB", "SOMA TOTAL")

0.3304229717459723

In [23]:
# 21. Análise temporal

df.groupBy("ANO") \
.agg(
    sum("SOMA TOTAL").alias("TOTAL")
) \
.orderBy(col("TOTAL").desc()) \
.show()

+----+-----+
| ANO|TOTAL|
+----+-----+
|2010| 4394|
|2011| 4388|
|2012| 3820|
|2016| 2907|
|2013| 2900|
|2014| 2754|
|2008| 2494|
|2004| 2400|
|2005| 2400|
|2006| 2395|
|2007| 2394|
|2015| 1940|
|2009| 1293|
|2017|  970|
+----+-----+



In [24]:
# 22. Últimos contingentes

df.orderBy(col("ANO").desc()).show()

+---------------------------------------------+----+-------+-------+---------+---+---+-------+-------+----------+-------+
|CONTROLE DE EFETIVO NO HAITI AT� MAIO DE 2016| ANO|BRABAT1|BRABAT2|BRAENGCOY| MB|FAB|ESTRANG|SOMA EB|SOMA TOTAL|POR ANO|
+---------------------------------------------+----+-------+-------+---------+---+---+-------+-------+----------+-------+
|                                 26� CONTBRAS|2017|    639|      0|      120|181| 30|      0|    759|       970|    970|
|                                 23� CONTBRAS|2016|    665|      0|      120|181|  4|      0|    785|       970|   NULL|
|                                 24� CONTBRAS|2016|    665|      0|      120|181|  4|      0|    785|       970|   NULL|
|                                 25� CONTBRAS|2016|    636|      0|      120|181| 30|      0|    756|       967|   2907|
|                                 22� CONTBRAS|2015|    665|      0|      120|181|  4|      0|    785|       970|   NULL|
|                       

In [32]:
from pyspark.sql.functions import sum, col

maior_ano = (
    df.groupBy("ANO")
      .agg(sum("SOMA TOTAL").alias("EFETIVO_TOTAL"))
      .orderBy(col("EFETIVO_TOTAL").desc())
)

maior_ano.show()

+----+-------------+
| ANO|EFETIVO_TOTAL|
+----+-------------+
|2010|         4394|
|2011|         4388|
|2012|         3820|
|2016|         2907|
|2013|         2900|
|2014|         2754|
|2008|         2494|
|2004|         2400|
|2005|         2400|
|2006|         2395|
|2007|         2394|
|2015|         1940|
|2009|         1293|
|2017|          970|
+----+-------------+



In [33]:
# Exportar os dados tratados

df.coalesce(1) \
.write \
.mode("overwrite") \
.option("header", True) \
.csv("haiti_tratado")

In [29]:
# Dados tratados para um grafico de Pizza

from pyspark.sql.functions import sum

totais = df.select(
    sum("BRABAT1").alias("BRABAT1"),
    sum("BRABAT2").alias("BRABAT2"),
    sum("BRAENGCOY").alias("BRAENGCOY"),
    sum("MB").alias("MB"),
    sum("FAB").alias("FAB"),
    sum("ESTRANG").alias("ESTRANG")
)

totais.show()

+-------+-------+---------+----+---+-------+
|BRABAT1|BRABAT2|BRAENGCOY|  MB|FAB|ESTRANG|
+-------+-------+---------+----+---+-------+
|  20987|   4772|     4624|6295|347|    424|
+-------+-------+---------+----+---+-------+



In [30]:
dados = [
    ("BRABAT1", 20987),
    ("BRABAT2", 4772),
    ("BRAENGCOY", 4624),
    ("MB", 6295),
    ("FAB", 347),
    ("ESTRANG", 424)
]

pizza = spark.createDataFrame(dados, ["Forca", "Efetivo"])

In [31]:
pizza.coalesce(1)\
.write\
.mode("overwrite")\
.option("header", True)\
.csv("forcas_militares")